In [22]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [23]:
BASE_DIR = r"c:\Users\Administrator\Desktop\数模校赛\题目发布\赛题\2026_A题"
DATA_PATH = os.path.join(BASE_DIR, "outputs", "data_smoothed.csv")
MODEL_PATH = os.path.join(BASE_DIR, "outputs", "xgboost_final_model.pkl")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [24]:
def choose_k_by_silhouette(x_scaled, k_min=2, k_max=6, random_state=42):
    best_k, best_score = 3, -1.0
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = km.fit_predict(x_scaled)
        score = silhouette_score(x_scaled, labels)
        if score > best_score:
            best_k, best_score = k, score
    return best_k, best_score

In [25]:
def build_feature_row(feature_names, u_list, t_list, temp, c_in, q, deutsch):
    row = {c: 0.0 for c in feature_names}
    for col in feature_names:
        for i in range(1, 5):
            if f"U{i}" in col:
                row[col] = u_list[i - 1]
            if f"T{i}" in col:
                row[col] = t_list[i - 1]
        if "Temp" in col:
            row[col] = temp
        if "C_in" in col:
            row[col] = c_in
        if "Q_Nm3h" in col:
            row[col] = q
        if "Deutsch" in col:
            row[col] = deutsch
    return pd.DataFrame([row], columns=feature_names)

In [26]:
def predict_cout(lr_model, xgb_res, feature_names, env_cond, u_list, t_list):
    d_fixed, k2_fixed = 0.5, 0.3
    omega_total = 0.0
    for i in range(4):
        omega_total += u_list[i] ** 2 / (d_fixed + k2_fixed * t_list[i]) ** 2
    deutsch = omega_total / env_cond["Q_Nm3h_smooth"]

    x_input = build_feature_row(
        feature_names=feature_names,
        u_list=u_list,
        t_list=t_list,
        temp=env_cond["Temp_C_smooth"],
        c_in=env_cond["C_in_gNm3_smooth"],
        q=env_cond["Q_Nm3h_smooth"],
        deutsch=deutsch,
    )
    y_phys = lr_model.predict([[deutsch]])[0]
    y_res = xgb_res.predict(x_input)[0]
    return np.exp(y_phys + y_res) * env_cond["C_in_gNm3_smooth"] * 1000.0


In [27]:
def sample_cluster_env(cluster_df, n_samples=20, random_state=42):
    cols = ["C_in_gNm3_smooth", "Temp_C_smooth", "Q_Nm3h_smooth"]
    rng = np.random.default_rng(random_state)
    mu = cluster_df[cols].mean().values
    sigma = cluster_df[cols].std().replace(0, 1e-6).values

    samples = []
    for _ in range(n_samples):
        x = rng.normal(mu, sigma)
        samples.append(
            {
                "C_in_gNm3_smooth": max(1e-3, x[0]),
                "Temp_C_smooth": x[1],
                "Q_Nm3h_smooth": max(1.0, x[2]),
            }
        )
    return samples

In [28]:
def decode_particle(p):
    """
    粒子维度:
    p[0]: u_base (38~70)
    p[1]: t_base (40~240)
    p[2:6]: u_mul for 4 fields (0.82~1.05)
    p[6:10]: t_mul for 4 fields (0.8~2.6), 并做单调约束
    """
    u_base = p[0]
    t_base = p[1]
    u_mul = p[2:6]
    t_mul = p[6:10]

    u_raw = np.array([u_base * m for m in u_mul], dtype=float)
    # 强制电压前高后低（非增序），并给定工程下界
    u_monotone = np.maximum.accumulate(u_raw[::-1])[::-1]
    u_monotone = np.maximum(u_monotone, 35.0)
    u_list = [float(round(x, 2)) for x in u_monotone]
    t_raw = [float(t_base * m) for m in t_mul]
    # 强制振打周期单调不减，避免不合理乱序
    t_monotone = np.maximum.accumulate(np.array(t_raw))
    t_list = [float(round(max(20.0, x), 2)) for x in t_monotone]
    return u_list, t_list


In [29]:
def objective_with_constraints(
    p,
    env_samples,
    env_mean,
    lr_model,
    xgb_res,
    feature_names,
    c_limit=10.0,
):
    u_list, t_list = decode_particle(p)

    # 目标：电压功耗 + 振打成本 + 过长周期积灰惩罚
    power_proxy = float(np.sum(np.square(u_list)))
    rapping_cost = float(1200.0 * np.sum(1.0 / np.array(t_list)))
    long_t_penalty = float(np.sum(np.maximum(0.0, np.array(t_list) - 220.0) ** 2) * 0.04)
    base_obj = power_proxy + rapping_cost + long_t_penalty

    c_preds = []
    for env in env_samples:
        c_preds.append(predict_cout(lr_model, xgb_res, feature_names, env, u_list, t_list))
    c_arr = np.array(c_preds)
    c_mean = float(np.mean(c_arr))
    c_p90 = float(np.quantile(c_arr, 0.90))
    c_p95 = float(np.quantile(c_arr, 0.95))
    c_mean_env = float(predict_cout(lr_model, xgb_res, feature_names, env_mean, u_list, t_list))

    # 约束惩罚：必须满足国家超低排放标准（10 mg/Nm3）
    violation = max(0.0, c_mean - c_limit) + max(0.0, c_p90 - c_limit)
    # 软鲁棒项：再压一下P95，避免边缘解
    robust_violation = max(0.0, c_p95 - (c_limit + 0.5))
    penalty = 1e5 * violation + 5e4 * robust_violation

    return {
        "fitness": base_obj + penalty,
        "base_obj": base_obj,
        "power_proxy": power_proxy,
        "rapping_cost": rapping_cost,
        "long_t_penalty": long_t_penalty,
        "cout_mean": c_mean,
        "cout_p90": c_p90,
        "cout_p95": c_p95,
        "cout_mean_env": c_mean_env,
        "u_list": u_list,
        "t_list": t_list,
    }

In [30]:
def pso_optimize_cluster(
    cluster_df,
    lr_model,
    xgb_res,
    feature_names,
    c_limit=10.0,
    n_particles=24,
    n_iter=40,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    env_mean = {
        "C_in_gNm3_smooth": cluster_df["C_in_gNm3_smooth"].mean(),
        "Temp_C_smooth": cluster_df["Temp_C_smooth"].mean(),
        "Q_Nm3h_smooth": cluster_df["Q_Nm3h_smooth"].mean(),
    }
    env_samples = sample_cluster_env(cluster_df, n_samples=20, random_state=random_state)

    # [u_base, t_base, u_mul1..4, t_mul1..4]
    lb = np.array([40, 40, 0.95, 0.90, 0.85, 0.80, 0.80, 1.00, 1.20, 1.40], dtype=float)
    ub = np.array([70, 240, 1.05, 1.02, 0.98, 0.95, 1.20, 1.60, 2.00, 2.60], dtype=float)
    dim = len(lb)

    pos = rng.uniform(lb, ub, size=(n_particles, dim))
    vel = rng.uniform(-1.0, 1.0, size=(n_particles, dim))
    pbest_pos = pos.copy()
    pbest_fit = np.full(n_particles, np.inf)
    pbest_meta = [None] * n_particles

    gbest_pos = None
    gbest_fit = np.inf
    gbest_meta = None

    w_max, w_min = 0.90, 0.40
    c1, c2 = 1.7, 1.7

    for it in range(n_iter):
        w = w_max - (w_max - w_min) * (it / max(1, n_iter - 1))

        for i in range(n_particles):
            meta = objective_with_constraints(
                p=pos[i],
                env_samples=env_samples,
                env_mean=env_mean,
                lr_model=lr_model,
                xgb_res=xgb_res,
                feature_names=feature_names,
                c_limit=c_limit,
            )
            fit = meta["fitness"]

            if fit < pbest_fit[i]:
                pbest_fit[i] = fit
                pbest_pos[i] = pos[i].copy()
                pbest_meta[i] = meta

            if fit < gbest_fit:
                gbest_fit = fit
                gbest_pos = pos[i].copy()
                gbest_meta = meta

        r1 = rng.uniform(size=(n_particles, dim))
        r2 = rng.uniform(size=(n_particles, dim))
        vel = (
            w * vel
            + c1 * r1 * (pbest_pos - pos)
            + c2 * r2 * (gbest_pos - pos)
        )
        pos = pos + vel
        pos = np.clip(pos, lb, ub)

    if gbest_meta is None:
        return None

    return {
        "u_rec": "/".join([str(x) for x in gbest_meta["u_list"]]),
        "t_rec": "/".join([str(x) for x in gbest_meta["t_list"]]),
        "cout_mean": gbest_meta["cout_mean"],
        "cout_p90": gbest_meta["cout_p90"],
        "cout_p95": gbest_meta["cout_p95"],
        "cout_mean_env": gbest_meta["cout_mean_env"],
        "power_proxy": gbest_meta["power_proxy"],
        "rapping_cost": gbest_meta["rapping_cost"],
        "long_t_penalty": gbest_meta["long_t_penalty"],
        "objective": gbest_meta["base_obj"],
        "fitness": gbest_meta["fitness"],
    }

In [ ]:
def main():
    print("读取数据与模型...")
    df = pd.read_csv(DATA_PATH)
    with open(MODEL_PATH, "rb") as f:
        models = pickle.load(f)
    lr_model = models["phys"]
    xgb_res = models["res"]
    feature_names = list(xgb_res.get_booster().feature_names)

    cluster_features = ["C_in_gNm3_smooth", "Temp_C_smooth", "Q_Nm3h_smooth"]
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(df[cluster_features])

    k, sil = choose_k_by_silhouette(x_scaled, k_min=2, k_max=6, random_state=42)
    print(f"KMeans自动选K: {k}, silhouette={sil:.4f}")
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    df["cluster"] = km.fit_predict(x_scaled)

    print("\n开始逐工况PSO优化（约束：Cout均值与P90 <= 10 mg/Nm3）...")
    records = []
    for cid in sorted(df["cluster"].unique()):
        cdf = df[df["cluster"] == cid].copy()
        print(f"\n正在优化工况{cid}，样本数={len(cdf)} ...")
        best = pso_optimize_cluster(
            cluster_df=cdf,
            lr_model=lr_model,
            xgb_res=xgb_res,
            feature_names=feature_names,
            c_limit=10.0,
            n_particles=24,
            n_iter=40,
            random_state=42 + int(cid),
        )
        if best is None:
            records.append(
                {
                    "cluster": int(cid),
                    "count": len(cdf),
                    "cin_mean": cdf["C_in_gNm3_smooth"].mean(),
                    "temp_mean": cdf["Temp_C_smooth"].mean(),
                    "q_mean": cdf["Q_Nm3h_smooth"].mean(),
                    "u_rec": "无可行解",
                    "t_rec": "无可行解",
                }
            )
            continue

        records.append(
            {
                "cluster": int(cid),
                "count": len(cdf),
                "cin_mean": cdf["C_in_gNm3_smooth"].mean(),
                "temp_mean": cdf["Temp_C_smooth"].mean(),
                "q_mean": cdf["Q_Nm3h_smooth"].mean(),
                **best,
            }
        )
        print(
            f"工况{cid}: U={best['u_rec']}, T={best['t_rec']}, "
            f"Cout均值={best['cout_mean']:.3f}, P90={best['cout_p90']:.3f}"
        )

    out_df = pd.DataFrame(records).sort_values("cin_mean", ascending=False).reset_index(drop=True)
    out_csv = os.path.join(OUTPUT_DIR, "optimization_results_q2_kmeans_pso.csv")
    out_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"\n已保存: {out_csv}")
    print("\n优化结果表:")
    print(out_df.to_string(index=False))


if __name__ == "__main__":
    main()


读取数据与模型...
KMeans自动选K: 6, silhouette=0.4392

开始逐工况PSO优化（约束：Cout均值与P90 <= 10 mg/Nm3）...

正在优化工况0，样本数=1250 ...
工况0: U=38.0/36.0/35.0/35.0, T=48.0/64.0/80.0/104.0, Cout均值=3.652, P90=3.852

正在优化工况1，样本数=2636 ...
工况1: U=38.0/36.0/35.0/35.0, T=62.55/78.19/93.82/109.46, Cout均值=7.946, P90=9.806

正在优化工况2，样本数=2889 ...
工况2: U=38.0/36.0/35.0/35.0, T=32.0/64.0/80.0/104.0, Cout均值=0.486, P90=0.574

正在优化工况3，样本数=120 ...
工况3: U=38.0/36.0/35.0/35.0, T=66.67/83.34/100.01/116.68, Cout均值=9.582, P90=10.000

正在优化工况4，样本数=1584 ...
工况4: U=38.0/36.0/35.0/35.0, T=65.35/87.14/87.14/141.6, Cout均值=8.281, P90=9.997

正在优化工况5，样本数=1551 ...


In [ ]:
# === 新增：聚类结果可视化代码 ===
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 7))
# 使用刚才分好的 'cluster' 列作为颜色标签
# 选取入口浓度 C_in 和温度 Temp_C 作为坐标轴
sns.scatterplot(
    data=df, 
    x="C_in_gNm3_smooth", 
    y="Temp_C_smooth", 
    hue="cluster", 
    palette="viridis", 
    alpha=0.6, 
    edgecolor='w'
)

plt.title(f'KMeans 聚类工况分布图 (K={k})', fontsize=15)
plt.xlabel('入口粉尘浓度 (g/Nm³)', fontsize=12)
plt.ylabel('烟气温度 (℃)', fontsize=12)
plt.legend(title='工况类别')
plt.grid(True, linestyle='--', alpha=0.5)

# 保存图片
plot_path = os.path.join(OUTPUT_DIR, "kmeans_cluster_distribution.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"聚类分布图已保存至: {plot_path}")
plt.show()

IndentationError: unexpected indent (1595780812.py, line 2)